# SQL im Browser mit JupySQL

Mit [JupySQL](https://jupysql.ploomber.io) schreibst du **reines SQL** in eine Notebook-Zelle – ohne Python-Drumherum. Alles läuft dabei ausschließlich in deinem Browser: Es gibt keinen Server, auf den Daten hochgeladen werden.

Als Datenbank dient eine kleine **SQLite**-Kinderbibliothek mit vier Tabellen:

| Tabelle | Inhalt |
| --- | --- |
| `authors` | Autorinnen und Autoren (`id`, `name`, `country`) |
| `books` | Bücher (`id`, `title`, `author_id`, `year`, `genre`, `pages`, `age_from`, `copies`) |
| `members` | Mitglieder der Bibliothek (`id`, `name`, `class`, `joined_on`) |
| `loans` | Ausleihen (`id`, `book_id`, `member_id`, `borrowed_on`, `returned_on`) |

Die Bücher und Namen sind frei erfunden. Der Bauplan der Datenbank steht in [`data/library.sql`](data/library.sql).

## 1. JupySQL installieren

Pakete landen hier nicht auf der Festplatte, sondern im Browser-Tab. Nach jedem Neuladen der Seite führst du diese Zelle also einmal aus.

In [ ]:
%pip install jupysql

## 2. Verbindung herstellen

`%load_ext sql` schaltet die SQL-Magics frei, `%sql` verbindet die Datenbank. `sqlite:///data/library.db` besteht aus zwei Teilen: `sqlite://` sagt, welche Art von Datenbank gemeint ist, `data/library.db` ist der Pfad zur Datei – genau so, wie sie links im Dateibrowser liegt.

In [ ]:
import warnings

warnings.filterwarnings("ignore", category=SyntaxWarning)  # JupySQL's import is noisy

%load_ext sql
%sql sqlite:///data/library.db

Ein kurzer Blick auf die Tabellen, die in der Datenbank stecken:

In [ ]:
%sqlcmd tables

In [ ]:
%sqlcmd columns --table books

## 3. Die erste Abfrage

`%sql` am Zeilenanfang führt eine einzelne SQL-Zeile aus.

In [ ]:
%sql SELECT title, year, genre FROM books ORDER BY year DESC LIMIT 5

## 4. Mehrzeilige Abfragen mit `%%sql`

Steht `%%sql` in der **ersten Zeile** einer Zelle, gilt die ganze Zelle als SQL. So lassen sich längere Abfragen lesbar formatieren – hier ein `JOIN` über zwei Tabellen.

In [ ]:
%%sql
SELECT b.title,
       a.name AS author,
       a.country,
       b.age_from
FROM books AS b
JOIN authors AS a ON a.id = b.author_id
WHERE b.age_from <= 6
ORDER BY b.age_from, b.title

## 5. Gruppieren und zählen

`GROUP BY` fasst Zeilen zusammen, die Aggregatfunktionen `COUNT`, `SUM`, `AVG`, `MIN` und `MAX` rechnen je Gruppe.

In [ ]:
%%sql
SELECT genre,
       COUNT(*) AS titel,
       ROUND(AVG(pages), 1) AS seiten_im_schnitt,
       MAX(year) AS neuestes_jahr
FROM books
GROUP BY genre
ORDER BY titel DESC, genre

## 6. Was ist gerade ausgeliehen?

In `loans` steht in `returned_on` ein Datum – oder `NULL`, wenn das Buch noch nicht zurück ist. Auf `NULL` prüft man mit `IS NULL`, nicht mit `= NULL`.

In [ ]:
%%sql
SELECT m.name AS mitglied,
       m.class AS klasse,
       b.title AS buch,
       l.borrowed_on AS ausgeliehen_am
FROM loans AS l
JOIN books AS b ON b.id = l.book_id
JOIN members AS m ON m.id = l.member_id
WHERE l.returned_on IS NULL
ORDER BY l.borrowed_on

## 7. Ergebnisse weiterverarbeiten

Bis hierher kam SQL ohne jede Zusatzbibliothek aus. Für das nächste Stück brauchen wir `pandas`: Damit wird aus einer Ergebnistabelle ein *DataFrame*, mit dem sich in Python weiterrechnen und zeichnen lässt.

`pandas` ist ein großes Paket und wird beim Ausführen der nächsten Zelle aus dem Netz nachgeladen – das dauert einen Moment. Wer nur SQL üben will, kann ab hier aufhören und bei den Aufgaben weitermachen.

In [ ]:
import pandas  # loaded on demand; everything above works without it

result = %sql SELECT genre, COUNT(*) AS titel FROM books GROUP BY genre ORDER BY titel DESC
df = result.DataFrame()
df

In [ ]:
import matplotlib.pyplot as plt

ax = df.plot.barh(x="genre", y="titel", legend=False, color="#4c72b0")
ax.set_xlabel("Anzahl Titel")
ax.set_ylabel("")
ax.set_title("Bücher je Genre")
plt.tight_layout()
plt.show()

## 8. Python-Werte in SQL einsetzen

Eine Python-Variable lässt sich mit doppelten geschweiften Klammern in die Abfrage einsetzen. So kannst du dieselbe Abfrage mit unterschiedlichen Werten laufen lassen, ohne den SQL-Text zu ändern.

In [ ]:
min_year = 2019

%sql SELECT title, year FROM books WHERE year >= {{min_year}} ORDER BY year

## 9. Selbst Daten schreiben

Bisher haben wir nur gelesen. Fürs Schreiben gibt es eine Besonderheit: Die Dateien im Dateibrowser liegen nicht auf einer echten Festplatte, sondern im Speicher des Browsers. Lesen klappt dort, aber SQLite braucht zum Schreiben mehr, als dieser Speicher bietet – die Datenbank würde beim ersten `INSERT` beschädigt.

Der Ausweg ist eine Kopie in `/tmp`, dem Arbeitsverzeichnis des Kernels. Dort verhält sich die Datei wie auf einer normalen Festplatte. Nebenbei bleibt so die Originaldatei unverändert: Beim nächsten Neustart des Kernels startest du wieder sauber.

In [ ]:
import shutil

shutil.copy("data/library.db", "/tmp/library.db")

Jetzt verbinden wir uns mit der Kopie. Der vierte Schrägstrich ist kein Tippfehler: Drei gehören zum `sqlite://`, der vierte ist der Anfang des absoluten Pfads `/tmp/library.db`.

In [ ]:
%sql sqlite:////tmp/library.db

In dieser Kopie darfst du alles: neue Tabellen anlegen, Zeilen einfügen, löschen.

In [ ]:
%%sql
CREATE TABLE IF NOT EXISTS wishlist (
    id    INTEGER PRIMARY KEY,
    title TEXT NOT NULL,
    votes INTEGER NOT NULL DEFAULT 0
);

INSERT INTO wishlist (title, votes) VALUES
    ('Der Keller der tausend Kekse', 7),
    ('Frieda funkt zum Mars', 4),
    ('Das Fahrrad, das Angst vor Bergen hatte', 9);

In [ ]:
%sql SELECT * FROM wishlist ORDER BY votes DESC

Auch andersherum geht es: Ein DataFrame lässt sich als Tabelle in die Datenbank schreiben. Das erledigt `to_sql` aus pandas.

(JupySQL hat dafür auch eine eigene Abkürzung, `%sql --persist`. Die funktioniert allerdings nur, wenn `pandas` schon geladen war, bevor `%load_ext sql` lief – und genau das wollen wir hier ja vermeiden.)

In [ ]:
import sqlite3

import pandas as pd

opening_hours = pd.DataFrame(
    {
        "day": ["Montag", "Dienstag", "Mittwoch", "Donnerstag", "Freitag"],
        "opens": ["09:00", "09:00", "13:00", "09:00", "09:00"],
        "closes": ["16:00", "16:00", "18:00", "16:00", "14:00"],
    }
)

with sqlite3.connect("/tmp/library.db") as con:
    opening_hours.to_sql("opening_hours", con, if_exists="replace", index=False)

In [ ]:
%sql SELECT * FROM opening_hours WHERE closes > '16:00'

## 10. Aufgaben

Probier die folgenden Abfragen selbst – die Zelle darunter ist dein Notizblock.

1. Welche Bücher haben mehr als 150 Seiten? Gib Titel und Seitenzahl aus, absteigend sortiert.
2. Wie viele Bücher hat jede Autorin bzw. jeder Autor geschrieben? (`JOIN` + `GROUP BY`)
3. Welches Mitglied hat bisher am häufigsten etwas ausgeliehen?
4. Welche Bücher wurden noch **nie** ausgeliehen? (Tipp: `LEFT JOIN` und `IS NULL` – oder `NOT IN`)
5. Wie viele Tage war jede zurückgegebene Ausleihe unterwegs? (Tipp: `julianday(returned_on) - julianday(borrowed_on)`)
6. Welche Klasse liest am meisten? Zähle die Ausleihen je `class`.

In [ ]:
%%sql
-- Aufgabe 1: Bücher mit mehr als 150 Seiten
SELECT title, pages
FROM books
-- WHERE ...
ORDER BY title

## Weiterlesen

- JupySQL-Dokumentation: https://jupysql.ploomber.io
- SQLite-Sprachreferenz: https://sqlite.org/lang.html
- Alle Magics auf einen Blick: `%sql?`, `%sqlcmd?` und `%config SqlMagic`